In [1]:

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path
import csv

# 한국 날짜를 기준으로 계산
today = datetime.now(ZoneInfo("Asia/Seoul")).date()

# 학생이 자신의 과제 정보로 변경할 부분
assignments = [
    {
        "course": "인공지능",
        "task": "AI Agent 프로젝트 보고서",
        "due": today + timedelta(days=2),
        "hours": 6,
        "progress": 20
    },
    {
        "course": "자료구조",
        "task": "알고리즘 구현 과제",
        "due": today + timedelta(days=5),
        "hours": 8,
        "progress": 50
    },
    {
        "course": "데이터베이스",
        "task": "SQL 실습 보고서",
        "due": today + timedelta(days=1),
        "hours": 3,
        "progress": 0
    },
    {
        "course": "컴퓨터네트워크",
        "task": "기말시험 대비",
        "due": today + timedelta(days=10),
        "hours": 12,
        "progress": 25
    }
]

# 마감일에 따른 우선순위 판정
def get_priority(days_left, progress):
    if progress >= 100:
        return 4, "✅ 완료"
    if days_left < 0:
        return 0, "🚨 기한 경과"
    if days_left <= 2:
        return 1, "🔴 긴급"
    if days_left <= 5:
        return 2, "🟠 높음"
    return 3, "🟢 보통"

results = []

for item in assignments:
    days_left = (item["due"] - today).days
    rank, priority = get_priority(
        days_left, item["progress"]
    )

    remaining_hours = round(
        item["hours"] * (1 - item["progress"] / 100), 1
    )

    results.append({
        "과목": item["course"],
        "과제": item["task"],
        "마감일": item["due"].isoformat(),
        "D-day": days_left,
        "남은시간": remaining_hours,
        "진행률": item["progress"],
        "우선순위": priority,
        "_rank": rank
    })

# 우선순위가 높은 과제부터 정렬
results.sort(
    key=lambda x: (x["_rank"], x["D-day"])
)

print("📚 대학생 과제·시험 관리 도우미")
print("오늘 날짜:", today)
print("=" * 55)

for row in results:
    print(f"\n{row['우선순위']} | {row['과목']}")
    print("과제:", row["과제"])
    print("마감:", row["마감일"])
    print("D-day:", row["D-day"])
    print("남은 예상 작업시간:", row["남은시간"], "시간")

# CSV 저장
output_dir = (
    Path("/workspace/results")
    if Path("/workspace").exists()
    else Path.cwd()
)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / (
    f"student_planner_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

fields = [
    "과목", "과제", "마감일", "D-day",
    "남은시간", "진행률", "우선순위"
]

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(
        {key: row[key] for key in fields}
        for row in results
    )

print("\n✅ 분석 완료!")
print("저장 파일:", output_file)


📚 대학생 과제·시험 관리 도우미
오늘 날짜: 2026-11-26

🔴 긴급 | 데이터베이스
과제: SQL 실습 보고서
마감: 2026-11-27
D-day: 1
남은 예상 작업시간: 3.0 시간

🔴 긴급 | 인공지능
과제: AI Agent 프로젝트 보고서
마감: 2026-11-28
D-day: 2
남은 예상 작업시간: 4.8 시간

🟠 높음 | 자료구조
과제: 알고리즘 구현 과제
마감: 2026-12-01
D-day: 5
남은 예상 작업시간: 4.0 시간

🟢 보통 | 컴퓨터네트워크
과제: 기말시험 대비
마감: 2026-12-06
D-day: 10
남은 예상 작업시간: 9.0 시간

✅ 분석 완료!
저장 파일: /workspace/results/student_planner_20261126_071306.csv


In [4]:

import requests

BASE_URL = "http://127.0.0.1:8000"

# 앞 단계에서 계산한 우선순위 높은 과제 두 개
top_tasks = results[:2]

task_text = "\n".join(
    f"{r['과목']}: {r['과제']}, "
    f"마감까지 {r['D-day']}일, "
    f"남은 작업 {r['남은시간']}시간"
    for r in top_tasks
)

question = (
    "다음 두 과제에 대해 오늘 무엇부터 해야 할지 "
    "한국어로 두 문장만 간단히 조언해줘.\n"
    + task_text
)

try:
    response = requests.post(
        BASE_URL + "/agent",
        json={"query": question},
        timeout=240
    )
    response.raise_for_status()

    data = response.json()

    print("실행 상태:", data.get("status"))

    if data.get("status") == "success":
        print("✅ AI 학습 코치")
        print(data.get("final_answer"))
    else:
        print("⚠️ AI 판단 오류")
        print(data.get("decision"))

except requests.RequestException as e:
    print("❌ 서버 요청 오류:", e)


실행 상태: success
✅ AI 학습 코치
데이터베이스 실습 보고서를 먼저 시작해 주세요. 인공지능 프로젝트 보고서는 다음날에 집중해서 처리할 수 있습니다.
